# 🧪 Test Scrapers
Test `scraper_landing.py` và `scraper_g2.py` trước khi tích hợp vào pipeline chính.

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5

---
## CELL 1 — Cài thư viện + Mount Drive

In [ ]:
!pip install requests beautifulsoup4 -q

from google.colab import drive
import sys

drive.mount('/content/drive')

# Thêm đường dẫn để import scraper files
PROJ_DIR = '/content/drive/MyDrive/competitor-intel-v2'
sys.path.append(PROJ_DIR)

print('✅ Ready')
print(f'📁 Project dir: {PROJ_DIR}')

---
## CELL 2 — Kiểm tra files có trong Drive không

In [ ]:
import os

required_files = ['scraper_landing.py', 'scraper_g2.py']

print('Kiểm tra files trong Drive:')
all_ok = True
for f in required_files:
    path   = os.path.join(PROJ_DIR, f)
    exists = os.path.exists(path)
    icon   = '✅' if exists else '❌'
    print(f'  {icon} {f}')
    if not exists:
        all_ok = False

if not all_ok:
    print('\n⚠️  Upload file còn thiếu vào Drive trước khi chạy tiếp.')
else:
    print('\n✅ Tất cả files đã sẵn sàng.')

---
## CELL 3 — Test scraper_landing.py
Test từng competitor một, in kết quả rõ ràng để kiểm tra.

In [ ]:
from scraper_landing import scrape_landing
import time

# ✏️ Thêm/bớt URL tại đây
TEST_URLS = [
    ('Amplitude'   , 'https://amplitude.com'),
    ('Mixpanel'    , 'https://mixpanel.com'),
    ('PostHog'     , 'https://posthog.com'),
    ('Heap'        , 'https://heap.io'),
    ('Pendo'       , 'https://www.pendo.io'),
]

print('🌐 TEST — scraper_landing.py')
print('=' * 60)

landing_results = {}
issues          = []

for name, url in TEST_URLS:
    print(f'\n[{name}]  {url}')
    result = scrape_landing(url)
    landing_results[name] = result

    # In kết quả
    h   = result['headline']
    sub = result['subheadline']
    cta = result['cta']

    print(f'  Headline    : {h[:75] if h else "❌ TRỐNG"}')
    print(f'  Subheadline : {sub[:75] if sub else "⚠️  TRỐNG"}')
    print(f'  CTA         : {cta if cta else "⚠️  TRỐNG"}')

    # Ghi nhận vấn đề
    if not h:
        issues.append(f'{name}: headline trống — trang có thể dùng JS')

    time.sleep(2)  # tránh rate limit

# Tổng kết
print('\n' + '=' * 60)
success = sum(1 for r in landing_results.values() if r['headline'])
print(f'\n📊 Kết quả: {success}/{len(TEST_URLS)} có headline')

if issues:
    print('\n⚠️  Vấn đề cần xử lý:')
    for issue in issues:
        print(f'   • {issue}')
else:
    print('✅ Tất cả OK!')

---
## CELL 4 — Test scraper_g2.py
Test 3-layer fallback: API → Capterra → Hardcoded.

In [ ]:
from scraper_g2 import get_reviews_data

# ✏️ Thêm/bớt slug tại đây (slug = phần cuối URL trên G2)
TEST_SLUGS = [
    'amplitude',
    'mixpanel',
    'posthog',
    'heap',
    'pendo',
]

print('⭐ TEST — scraper_g2.py')
print('=' * 60)

g2_results = {}
issues     = []

for slug in TEST_SLUGS:
    print(f'\n[{slug}]')
    result = get_reviews_data(slug)
    g2_results[slug] = result

    rating = result['avg_rating']
    count  = result['review_count']
    source = result['source']

    # Màu theo source
    source_icon = {
        'g2_api'            : '🟢 G2 API',
        'capterra'          : '🔵 Capterra',
        'hardcoded_q4_2024' : '🟡 Hardcoded',
        'failed'            : '🔴 Failed',
    }.get(source, f'⚪ {source}')

    print(f'  Rating  : {rating}⭐')
    print(f'  Reviews : {count:,}')
    print(f'  Source  : {source_icon}')

    if rating == 0:
        issues.append(f'{slug}: rating = 0 — kiểm tra lại slug')

    time.sleep(2)

# Tổng kết
print('\n' + '=' * 60)
success = sum(1 for r in g2_results.values() if r['avg_rating'] > 0)
sources = {r['source'] for r in g2_results.values()}
print(f'\n📊 Kết quả: {success}/{len(TEST_SLUGS)} có rating')
print(f'   Sources used: {", ".join(sources)}')

if issues:
    print('\n⚠️  Vấn đề:')
    for issue in issues:
        print(f'   • {issue}')
else:
    print('\n✅ Tất cả OK!')

---
## CELL 5 — Tổng hợp kết quả
Gộp landing + G2 data lại, xem combined output trước khi đưa vào pipeline chính.

In [ ]:
print('📋 COMBINED OUTPUT')
print('=' * 60)

for name in [n for n, _ in TEST_URLS]:
    slug   = name.lower()
    land   = landing_results.get(name, {})
    review = g2_results.get(slug, {})

    combined = {**land, **review}

    # Status checks
    headline_ok = '✅' if combined.get('headline') else '❌'
    rating_ok   = '✅' if combined.get('avg_rating', 0) > 0 else '⚠️ '

    print(f'\n{name}')
    print(f'  {headline_ok} Headline : {str(combined.get("headline",""))[:65]}')
    print(f'     CTA      : {combined.get("cta", "—")}')
    print(f'  {rating_ok} G2 Rating: {combined.get("avg_rating",0)}⭐  ({combined.get("review_count",0):,} reviews)  [{combined.get("source","?")}]')

print('\n' + '=' * 60)
print('\n✅ Data sẵn sàng để đưa vào pipeline chính.')
print('   Bước tiếp: chạy competitor_intel_v2.ipynb Cell 10')